# testing demucs adapter

In [1]:
from src.adapters.DemucsAdapter import DemucsAdapter
demucs = DemucsAdapter(source_path="./input/on tha line.mp3",
                       output_dir="./output/demucs_test",
                       demucs_model="htdemucs")
demucs_result = demucs.run()

Running: /envs/karaoke_gen_pydemo2/bin/python -m demucs -n htdemucs --two-stems vocals -o output/demucs_test/separated input/on tha line.mp3
Selected model is a bag of 1 models. You will see that many progress bars per track.
Separated tracks will be stored in /home/matej/Documents/Projects/karaoke_gen_pydemo2/testing/output/demucs_test/separated/htdemucs
Separating track input/on tha line.mp3


100%|████████████████████████████████████████████████████████████████████████| 175.5/175.5 [00:06<00:00, 27.95seconds/s]


In [2]:
print(demucs_result)

SourceSeparationResult(source_path='input/on tha line.mp3', vocals_path='output/demucs_test/separated/htdemucs/on tha line/vocals.wav', instrumental_path='output/demucs_test/separated/htdemucs/on tha line/no_vocals.wav')


# Initial transcription whith whisperX

In [3]:
from src.adapters.WhisperXAdapter import WhisperXAdapter
whisper = WhisperXAdapter(model_name="medium", language="en", batch_size=128)
whisper_result = whisper.transcribe(demucs_result.vocals_path)

2026-09-22 14:38:13 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../.local/lib/python3.11/site-packages/whisperx/assets/pytorch_model.bin`
/home/matej/.local/lib/python3.11/site-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(


In [4]:
print(whisper_result)

Transcription(language='en', segments=[TranscriptSegment(index=0, text="Yeah, how the he does it? Stuck in the me's and me's fields, yeah, cause I love it. 45 seconds, you a n***a off, who does it? Yeah, and it's way too far, gonna save that shit, I won't touch it. Beggin' for shot, gonna break, kick back like buckets. Snappin' that face in my circle like Kit Kat does it. I could take a lil' bitch if I want to, yeah, but I doesn't.", start_ms=15252, end_ms=36650, confidence=None, words=None), TranscriptSegment(index=1, text="I just chatted with G.C. if I took a shit at the shop to go later to my house They asked for the hell I made money, I couldn't even tell them cause I don't know how Turn that D.B. till it blazing, yeah Turn that D.B. till it bound You a too far girl, I couldn't say that shit, so pray that you're on the coach Big dough, yeah you most Can't hear you rot with it, you not a boss I done invented this shit on my soul And I go rock out of every show You not my twin, you n

# Genius adapter test

In [5]:
from src.adapters.GeniusLyricsAdapter import GeniusLyricsAdapter
from src.config import load_settings
from src.domain.SongMetadata import SongMetadata
from src.services.GeniusLyricsNormalizer import GeniusLyricsNormalizer
from src.domain.normalization.LyricsNormalizationStrategy import RemoveBracketedAnnotationsStrategy

settings = load_settings()

adapter = GeniusLyricsAdapter(
    access_token=settings.genius_access_token,
    timeout_seconds=settings.genius_timeout_seconds,
    retries=settings.genius_retries,
    sleep_time_seconds=settings.genius_sleep_time_seconds,
)

result = adapter.find_lyrics(
    SongMetadata(
        title="on tha line",
        artist="yeat",
    )
)

normalizer = GeniusLyricsNormalizer([RemoveBracketedAnnotationsStrategy()])
normalized_result = normalizer.normalize(result)

if result is None:
    print("No Genius lyrics were found.")
else:
    print(f"\nnormalized lyrics:\n{normalized_result.normalized_text}")


normalized lyrics:
Yeah, how does he does it? (Ah, yeah)
Stuck in the mix, mix pills, yeah, 'cause I love it (Phew, yeah)
Forty-five seconds, you in and you out for a dozen (Phew)
Yeah, and it's way too far gone to save that shit, I won't touch it (Yeah)
Back of a shotgun barrel kick back like buckets
Snappin' the fakes in my circle like Kit-Kat does it
I could take your lil' bitch if I want to, yeah, but I doesn't (But I doesn't, hey)
I just chopped up the G65, took it straight out the shop, take it back to my house (Woo)
They ask how the hell I make money, I couldn't еven tell 'em 'causе I don't know how (Ah)
Turn up that beat 'til it's blazin', yeah, turn up the beat 'til it bounce (Yeah)
I'm too far gone, couldn't save that shit, so I'm Perc'ed out, yeah, on the couch (Yeah, yeah, yeah)
I'm a big dog, yeah, you a mouse (Yeah, yeah, yeah)
Can't even rock with you, you not a boss (Yeah, yeah, yeah)
I done invented this shit, that's my sauce (Yeah, yeah, yeah)
And I'm gon' rock out a

# Lyrics corrector test

In [6]:
from src.services.LyricsCorrector import LyricsCorrector
corrector = LyricsCorrector()

corrected_transcription = corrector.correct(whisper_result, normalized_result.normalized_text)

In [7]:
print(corrected_transcription)

LyricsCorrectionResult(transcription=Transcription(language='en', segments=[TranscriptSegment(index=0, text="Yeah, how does he does it? Stuck in the mix, mix pills, yeah, 'cause I love it Forty-five seconds, you in and you out for a dozen Yeah, and it's way too far gone to save that shit, I won't touch it Back of a shotgun barrel kick back like buckets Snappin' the fakes in my circle like Kit-Kat does it I could take your lil' bitch if I want to, yeah, but I doesn't", start_ms=15252, end_ms=36650, confidence=None, words=None), TranscriptSegment(index=1, text="I just chopped up the G65, took it straight out the shop, take it back to my house They ask how the hell I make money, I couldn't еven tell 'em 'causе I don't know how Turn up that beat 'til it's blazin', yeah, turn up the beat 'til it bounce I'm too far gone, couldn't save that shit, so I'm Perc'ed out, yeah, on the couch I'm a big dog, yeah, you a mouse Can't even rock with you, you not a boss I done invented this shit, that's m

# Align the corrected lyrics

In [8]:
aligned = whisper.align(demucs_result.vocals_path, corrected_transcription.transcription)

In [9]:
print(aligned)

Transcription(language='en', segments=[TranscriptSegment(index=0, text='Yeah, how does he does it?', start_ms=15312, end_ms=16813, confidence=None, words=[WordSegment(word='Yeah,', start_ms=15312, end_ms=15572, confidence=0.754), WordSegment(word='how', start_ms=16113, end_ms=16233, confidence=0.273), WordSegment(word='does', start_ms=16253, end_ms=16333, confidence=0.21), WordSegment(word='he', start_ms=16373, end_ms=16453, confidence=0.994), WordSegment(word='does', start_ms=16493, end_ms=16733, confidence=0.883), WordSegment(word='it?', start_ms=16753, end_ms=16813, confidence=0.327)]), TranscriptSegment(index=1, text="Stuck in the mix, mix pills, yeah, 'cause I love it Forty-five seconds, you in and you out for a dozen Yeah, and it's way too far gone to save that shit, I won't touch it Back of a shotgun barrel kick back like buckets Snappin' the fakes in my circle like Kit-Kat does it I could take your lil' bitch if I want to, yeah, but I doesn't", start_ms=18074, end_ms=36110, con

# Export lyrics

In [10]:
from src.domain.export.LyricsExportStrategy import SRTExportStrategy, JsonExportStrategy
from src.services.LyricsExportService import LyricsExportService

export_service = LyricsExportService()

In [11]:
export_service.export(aligned, "./output/lyrics_aligned.srt", SRTExportStrategy())

PosixPath('output/lyrics_aligned.srt')

In [13]:
export_service.export(aligned, "./output/lyrics_aligned.json", JsonExportStrategy())

PosixPath('output/lyrics_aligned.json')